In [ ]:
import scanpy as sc
import numpy as np
import random
import sys
sys.path.append("../../mapqc/")
import mapqc
import plotting

In [ ]:
%load_ext lab_black

### Set paths and parameters

In [ ]:
atlas_path = "../../data/HLCA_full_v1.1_emb.h5ad"

metadata key/cat info:

In [ ]:
ref_q_key = "core_or_extension"
ref_cat = "core"
query_cat = "extension"
condition_key = "lung_condition"
batch_key = "dataset"

parameters:

In [ ]:
n_nhoods = 100
k = 1000

### Prepare anndata:

read in full atlas (with all queries)

In [ ]:
adata_full = sc.read_h5ad(atlas_path)

#### choose query dataset and query condition of interest and subset data:

list reference batches:

In [ ]:
print("Reference batches:")
print(
    sorted(adata_full.obs.loc[adata_full.obs[ref_q_key] == ref_cat, batch_key].unique())
)

list query batches:

In [ ]:
print("Query batches:")
print(
    sorted(
        adata_full.obs.loc[adata_full.obs[ref_q_key] == query_cat, batch_key].unique()
    )
)

subset to query batch of choice:

In [ ]:
query_batch = "Zhang_2021"

In [ ]:
adata = adata_full[
    (adata_full.obs[ref_q_key] == ref_cat) | (adata_full.obs[batch_key] == query_batch)
].copy()

If using a reference batch as "simulated" query, annotate it as query here:

In [ ]:
if query_batch in (adata.obs.loc[adata.obs[ref_q_key] == ref_cat, batch_key].values):
    print(f"Setting batch {query_batch} to 'query' as query simulation.")
    adata.obs[ref_q_key] = adata.obs[ref_q_key].tolist()
    adata.obs.loc[adata.obs[batch_key] == query_batch, ref_q_key] = query_cat

Some clean-up for the Thienpont_2018 study specifically:

In [ ]:
if query_batch in ["Thienpont_2018_10Xv1", "Thienpont_2018_10Xv2"]:
    print("setting Thienpont tumor adjacent samples to 'Healthy'")
    lambrechts_sample_info = pd.read_csv(
        "./lambrechts_health_status_info.csv", index_col=0
    )
    lambrechts_healthy_samples = lambrechts_sample_info.loc[
        ~lambrechts_sample_info.cell_from_tumor, :
    ].index.tolist()
    adata.obs.loc[
        adata.obs["sample"].values.isin(lambrechts_healthy_samples), condition_key
    ] = "Healthy"

check which conditions are in the query:

In [ ]:
sorted(adata.obs.loc[adata.obs[batch_key] == query_batch, condition_key].unique())

Subset to specific condition for the query specifically (reference is already fully healthy):

In [ ]:
condition_of_interest = "Healthy"

In [ ]:
print(f"Subsetting query to condition {condition_of_interest}")
adata = adata[
    (adata.obs[ref_q_key] == ref_cat)
    | (adata.obs[condition_key] == condition_of_interest)
].copy()

### Run mapQC

randomly sample neighborhood center cells from query (this should probably be incorporated in the package as well, maybe some "prep_adata" function...)

In [ ]:
query_cell_idc = np.where(adata.obs[ref_q_key] == query_cat)[0].tolist()
random.seed(0)
center_cell_idc = random.sample(query_cell_idc, k=n_nhoods)
center_cells = np.array(adata.obs.iloc[center_cell_idc, :].index)

create mapqc results instance:

In [ ]:
results = mapqc.mapQC_results(
    adata=adata,
    center_cells=center_cells,
    k=k,
    ref_q_key="core_or_extension",
    ref_cat="core",
    query_cat="extension",
)

calculate sample distances to ref samples:

In [ ]:
results.calculate_sample_dists_to_ref_samples()

calculate stats on distances:

In [ ]:
results.calculate_nhood_statistics()

In [ ]:
results.nhood_stats

plot results:

In [ ]:
plotting.plot_boxplot_of_ref_vs_q_dists_per_nhood(results)

In [ ]:
plotting.plot_umap_significance(results)